# Environment Setup

In [7]:
import pandas as pd
# Loads the dataframe
df = pd.read_csv("dataset2.csv")

In [8]:
df.head()

,time,month,hours_after_sunset,bat_landing_number,food_availability,rat_minutes,rat_arrival_number
0,26/12/2017 16:13,0,-0.5,20,4.000000,0.0,0
1,26/12/2017 16:43,0,0.0,28,4.000000,0.0,0
2,26/12/2017 17:13,0,0.5,25,4.000000,0.0,0
3,26/12/2017 17:43,0,1.0,71,4.000000,0.0,0
4,26/12/2017 18:13,0,1.5,44,3.753857,0.0,0


# Missing Data Check

In [9]:
# Finds cells with no data and prints how many there are in each column 
missing_counts = df.isnull().sum()
print(missing_counts)

time                  0
month                 0
hours_after_sunset    0
bat_landing_number    0
food_availability     0
rat_minutes           0
rat_arrival_number    0
dtype: int64


# Chronology Check

In [10]:
# Converts the data in the "time" column into the correct datetime format
df["time"] = pd.to_datetime(df["time"], format="%d/%m/%Y %H:%M")

In [11]:
# Tests to see if the the data is chronologically ordered and prints the result.
chron_sorted = df["time"].is_monotonic_increasing
print("The 'time' column is sorted:", chron_sorted)

The 'time' column is sorted: True


# Standardise the Unit of Time Across the Dataset

In [12]:
df["minutes_after_sunset"] = df["hours_after_sunset"] * 60
df.drop(columns=["hours_after_sunset"], inplace=True)

# Create a new colum
Created a "date" column in both datasets to align bat landing with observation periods. 

In [13]:
# Creating new Column called Date
df['date'] = df['time'].dt.date

# Outlier Check

In [14]:

# Selects the columns to be check for outliers
outlier_cols = [
    "minutes_after_sunset",
    "bat_landing_number",
    "food_availability",
    "rat_minutes",
    "rat_arrival_number"
]

# Detects outliers using the IQR method
def is_outlier(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (series < lower_bound) | (series > upper_bound)

## Resolve Outliers

In [15]:
# Replaces outliers with the median amount of the column
for col in outlier_cols:
    outliers = is_outlier(df[col])
    median_value = df[col].median()
    df.loc[outliers, col] = median_value

# Save the Cleaned Data to a New Dataset

In [16]:
# Saves all data to "dataset2_cleaned.csv"
df.to_csv("dataset2_cleaned.csv", index=False)